<div dir="rtl" style="text-align: right; font-family: Tahoma, Vazirmatn, sans-serif; line-height: 1.8;">

<h1>تکلیف: پیاده‌سازی سیستم RAG و مقایسه استراتژی‌های Prompting</h1>

<h2>مقایسه استراتژی‌های Zero-shot، Few-shot و Chain-of-Thought در سیستم RAG</h2>

<hr>

<h2>راه‌های ارتباطی</h2>
<ul>
  <li>📧 <b>ایمیل:</b> <a href="mailto:erfanshahabi@ut.ac.ir">erfanshahabi@ut.ac.ir</a></li>
</ul>

<hr>

<h2>مجموعه‌داده (Dataset)</h2>

<p>
در این تکلیف از زیر مجموعه‌ای از مجموعه‌داده
<b>rajpurkar/squad</b>
و از معیارهای ارزیابی
<b>ROUGE (Recall-Oriented Understudy for Gisting Evaluation)</b>
و
شباهت معنایی
استفاده می‌کنیم.
</p>

<ul>
  <li>از مجموعه‌داده اصلی <b>SQuAD</b>، یک زیرمجموعه (<b>Subset</b>) انتخاب شده است:
    <ul>
      <li><b>100</b> پاراگراف (<b>Context</b>) از ویکی‌پدیا به عنوان corpus — فایل <b>squad_corpus_100.csv</b></li>
      <li><b>20</b> سوال و جواب (<b>Gold QA</b>) برای ارزیابی — فایل <b>squad_gold_20.csv</b></li>
      <li><b>4561</b> نمونه برای Fine-tuning — فایل <b>finetune_train.csv</b></li>
      <li><b>303</b> نمونه برای اعتبارسنجی Fine-tuning — فایل <b>finetune_val.csv</b></li>
    </ul>
  </li>
  <li>هر نمونه شامل:
    <ul>
      <li><b>context</b>: پاراگراف متنی از ویکی‌پدیا</li>
      <li><b>question</b>: سوال مرتبط با متن</li>
      <li><b>answer</b>: جواب gold به صورت span از متن</li>
    </ul>
  </li>
  <li>تمامی فایل‌های داده در فایل تمرین قرار داده شده‌اند.</li>
</ul>

<hr>

<h2>مدل‌ها (Models)</h2>

<p>
<b>Qwen/Qwen2.5-1.5B-Instruct, FacebookAI/roberta-large-mnli, BAAI/bge-small-en-v1.5</b>
</p>

<hr>

<h2>نکات مهم</h2>

<ul>
  <li>در پایان <b>هر مرحله</b>، نتایج را در یک فایل <b>CSV</b> ذخیره کنید و در فایل ارسالی پاسخ تمرین قرار دهید.</li>
  <li>فایل‌های CSV نتایج پاسخ مدل به سوالات در هر مرحله باید در فایل ارسالی پاسخ تمرین قرار داده شوند.</li>
  <li>فایل نهایی ارسالی باید یک فایل ZIP شامل همین نوت بوک و فایل پاسخ‌های مدل در هر مرحله در قال CSV باشد.</li>
  <li>مجموع نمرات این تمرین <b>57</b> نمره می‌باشد.</li>
</ul>

</div>

---
## 0  Install Requirements and import libraries and datasets

In [16]:
import os
import sys

if not os.path.exists("Data"):
    if not os.path.exists("a.zip"):
        !wget "https://filebin.net/lflr1ib29zexx0yz/a.zip"
    !unzip a.zip

In [17]:
# !pip install -r requirements.txt
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
  !pip install --upgrade transformers==5.12.1
if 'google.colab' in sys.modules:
  !pip install peft==0.18.1
!pip install trl==1.7.0 lancedb==0.33.0 rouge_score==0.1.2  transformers==5.12.1


/Users/morteza/.pyenv/versions/3.12.9/lib/python3.12/pty.py:95: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid, fd = os.forkpty()
Exception in thread LanceDBBackgroundEventLoop:
Traceback (most recent call last):
  File "/Users/morteza/.pyenv/versions/3.12.9/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/Users/morteza/.pyenv/versions/3.12.9/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/morteza/.pyenv/versions/3.12.9/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    self._run_once()
  File "/Users/morteza/.pyenv/versions/3.12.9/lib/python3.12/asyncio/base_events.py", line 1961, in _run_once
    event_list = self._selector.s

ERROR: Could not find an activated virtualenv (required).


In [18]:
# Write Your Code Here
# !uv pip install -r requirements.txt
import pandas as pd
import numpy as np
import transformers
import tokenizers
import lancedb
import torch
import torch.nn.functional as F
import random
from rich.table import Table
from rich.console import Console
import pyarrow as pa
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset as HuggingFaceDataset
from tqdm import tqdm
from rouge_score import rouge_scorer
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rouge_score import rouge_scorer
from tqdm import tqdm
from transformers import TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from trl import SFTTrainer

---
## 1  Define Schema and Store Embeddings in LanceDB (3 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, you need to define an appropriate Schema for storing text passages in LanceDB and use the embedding model
<b>BAAI/bge-small-en-v1.5</b>
to convert the passages from <b>squad_corpus_100.csv</b> into vector representations and store them in the database.
Then, run a sample semantic search query and display the top results to verify that the retrieval is working correctly.
</p>

In [19]:
# Write Your Code Here
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
print(device)

mps


In [20]:
IS_TEST = False
FINETUNE_DONE = True

In [21]:
class SquadDataset(Dataset):
    def __init__(self):
        df = pd.read_csv('./Data/squad_corpus_100.csv')
        df.drop(['Unnamed: 4'], axis = 1, inplace = True)
        df.set_index('id', inplace = True)
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        question = row['question']
        context = row['context']
        return question, context
dataset = SquadDataset()
dataloader = DataLoader(dataset, batch_size=10, shuffle=False)
dataset.df

,title,context,question
id,,,
1,"Jacksonville,_Florida",The area of the modern city of Jacksonville ha...,Who discovered pottery found on Black Hammock ...
2,Super_Bowl_50,Denver took the opening kickoff and started ou...,Who was at the receiving end of a 22-yard pass...
3,Prime_number,"After the Greeks, little happened with the stu...",Of what form do Mersenne primes take?
4,United_Methodist_Church,"Historically, the Methodist Church has support...",What does the United Methodist Church use in t...
5,Scottish_Parliament,Whilst the permanent building at Holyrood was ...,Where were interviews held while the parliamen...
...,...,...,...
96,Civil_disobedience,One of the oldest depictions of civil disobedi...,Who is Antigone's father in the play?
97,Intergovernmental_Panel_on_Climate_Change,The Intergovernmental Panel on Climate Change ...,What organization is the IPCC a part of?
98,Warsaw,John Paul II's visits to his native country in...,Where did John Paul II celebrate Mass in Warsaw?


In [ ]:
os.environ["HF_TOKEN"] = "TOKEN"
tokenizer = transformers.AutoTokenizer.from_pretrained('BAAI/bge-small-en-v1.5')
model = transformers.AutoModel.from_pretrained('BAAI/bge-small-en-v1.5').to(device)
print()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4693.73it/s]


In [23]:
for i in range(3):
    row = dataset.df.iloc[i]
    for col in ("question", "title", "context"):
        print(col + "  ", "\t", row[col])

    inputs = tokenizer(
        row["context"],
        return_tensors='pt',
        truncation=True,
        padding=True
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

    print("Embedding shape:", embeddings.shape)
    print("#######################" * 3)


question   	 Who discovered pottery found on Black Hammock Island?
title   	 Jacksonville,_Florida
context   	 The area of the modern city of Jacksonville has been inhabited for thousands of years. On Black Hammock Island in the national Timucuan Ecological and Historic Preserve, a University of North Florida team discovered some of the oldest remnants of pottery in the United States, dating to 2500 BC. In the 16th century, the beginning of the historical era, the region was inhabited by the Mocama, a coastal subgroup of the Timucua people. At the time of contact with Europeans, all Mocama villages in present-day Jacksonville were part of the powerful chiefdom known as the Saturiwa, centered around the mouth of the St. Johns River. One early map shows a village called Ossachite at the site of what is now downtown Jacksonville; this may be the earliest recorded name for that area.
Embedding shape: (1, 384)
#####################################################################
question   

In [24]:
db = lancedb.connect("./Data/lancedb")
schema = pa.schema(
    [
        ("id", pa.int64()),
        ("question", pa.string()),
        ("context", pa.string()),
        (
            "embedding",
            pa.fixed_shape_tensor(pa.float32(), (384,)),
        ),
    ]
)


try:
    db.drop_table("squad_embeddings")
except Exception as e:
    print(f"Error dropping table: {e}")
table = db.create_table(
    "squad_embeddings",
    schema=schema
)

In [25]:

def mean_pool(last_hidden_state, attention_mask):
    """ this function was written by AI when i asked for pooling suitable for BGE model"""
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

In [26]:
table.delete(where="id >= 0")
for batch in tqdm(dataloader):
    questions, contexts = batch
    inputs = tokenizer(
        contexts,
        return_tensors='pt',
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    embeddings = mean_pool(outputs.last_hidden_state, inputs['attention_mask']).cpu().numpy()
    rows = [{
        "id": len(table) + i,
        "question": questions[i],
        "context": contexts[i],
        "embedding": embeddings[i],
    } for i in range(len(questions))]
    table.add(rows)
print("table size", len(table))

100%|██████████| 10/10 [00:01<00:00,  6.08it/s]

table size 100


In [27]:
random.seed(42)

random_index = random.randint(0, len(dataset) - 1)
question, context = dataset[random_index]

inputs = tokenizer(
    ["query: " + question],
    return_tensors="pt",
    truncation=True,
    padding=True,
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

query_embedding = outputs.last_hidden_state[:, 0]
query_embedding = F.normalize(query_embedding, p=2, dim=1)
query_embedding = query_embedding.cpu().numpy()[0]

results = (
    table.search(
        query_embedding,
        vector_column_name="embedding"
    )
    .limit(1)
    .to_list()
)


print("Question:\t", question)
print("Context:\t", context)
print("Retrieved:\t", results[0]["context"])

Question:	 What did Martin Luther fear after a lightening bolt struck near him?
Context:	 He later attributed his decision to an event: on 2 July 1505, he was returning to university on horseback after a trip home. During a thunderstorm, a lightning bolt struck near him. Later telling his father he was terrified of death and divine judgment, he cried out, "Help! Saint Anna, I will become a monk!" He came to view his cry for help as a vow he could never break. He left law school, sold his books, and entered a closed Augustinian cloister in Erfurt on 17 July 1505. One friend blamed the decision on Luther's sadness over the deaths of two friends. Luther himself seemed saddened by the move. Those who attended a farewell supper walked him to the door of the Black Cloister. "This day you see me, and then, not ever again," he said. His father was furious over what he saw as a waste of Luther's education.
Retrieved:	 He later attributed his decision to an event: on 2 July 1505, he was returnin

---
## 2  Define a function for semantic search in lanceDB (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, you need to define a retrieval function that searches LanceDB using the questions from <b>squad_gold_20.csv</b>.
For each question, the function should return the top <b>5</b> most semantically similar passages from the database.
</p>

In [28]:
# Write Your Code Here
gold_df = pd.read_csv('./Data/squad_gold_20.csv')
gold_df

,id,title,question,answer
0,73,Super_Bowl_50,When is the last time a fumble return touchdow...,Super Bowl XXVIII
1,65,1973_oil_crisis,"According to the AAA, what is the percentage o...","last week of February 1974,"
2,17,French_and_Indian_War,What choice did French have for surrendering l...,continental North American possessions east of...
3,51,European_Union_law,Which articles state that the member states' r...,Articles 106 and 107
4,42,Computational_complexity_theory,The time required to output an answer on a det...,state transitions
5,10,Scottish_Parliament,How many general questions are available to op...,four
6,45,Civil_disobedience,What group of people performed revolutionary c...,Hungarians
7,47,Doctor_Who,What series featured Doctors from the revised ...,Destiny of the Doctor
8,35,American_Broadcasting_Company,When did Mark Woods leave ABC?,"June 30, 1951"
9,5,Scottish_Parliament,Where were interviews held while the parliamen...,courtyard


In [29]:
def retrieve_for_gold(question: str, k: int = 5):

    inputs = tokenizer(
        [question],
        return_tensors="pt",
        truncation=True,
        padding=True,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    query_embedding = outputs.last_hidden_state[:, 0]
    query_embedding = F.normalize(query_embedding, p=2, dim=1)
    query_embedding = query_embedding.cpu().numpy()[0]

    results = (
        table.search(
            query_embedding,
            vector_column_name="embedding"
        )
        .limit(k)
        .to_list()
    )


    return [i["context"] for i in results]
print(retrieve_for_gold(gold_df.iloc[1]["question"], k=5))


["Apollo 8 was planned to be the D mission in December 1968, crewed by McDivitt, Scott and Schweickart, launched on a Saturn V instead of two Saturn IBs. In the summer it had become clear that the LM would not be ready in time. Rather than waste the Saturn V on another simple Earth-orbiting mission, ASPO Manager George Low suggested the bold step of sending Apollo 8 to orbit the Moon instead, deferring the D mission to the next mission in March 1969, and eliminating the E mission. This would keep the program on track. The Soviet Union had sent animals around the Moon on September 15, 1968, aboard Zond 5, and it was believed they might soon repeat the feat with human cosmonauts. The decision was not announced publicly until successful completion of Apollo 7. Gemini veterans Frank Borman and James Lovell, and rookie William Anders captured the world's attention by making 10 lunar orbits in 20 hours, transmitting television pictures of the lunar surface on Christmas Eve, and returning saf

---
## 3  Loading Model (2 Points)

Load the **Qwen/Qwen3.5-2B** model and its tokenizer from 🤗 Hugging Face using `AutoModelForCausalLM` and `AutoTokenizer`.

In [30]:
# Write Your Code Here
model_name = 'qwen/qwen3.5-2b'
# model_name = 'Qwen/Qwen3-VL'
# model_name = "Qwen/Qwen3.5-2B-Instruct"
qwen_tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
qwen_model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto"
).to(device)
qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 8377.82it/s]


---
## 4  Search Without RAG (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, define a prompt-building function and an appropriate system prompt, then answer the 20 questions from <b>squad_gold_20.csv</b> using only the model's internal knowledge, without retrieving any passages from LanceDB.
Store the predicted answers alongside the gold answers for evaluation in later sections.
You have to print the model responses.
</p>

In [31]:
get_qwen_input_cache = {}

def get_qwen_output(prompt, max_new_tokens=64, model=None, tokenizer=None):
  if model is None:
    model = qwen_model
  if tokenizer is None:
    tokenizer = qwen_tokenizer

  cache_key = repr(prompt)
  if cache_key in get_qwen_input_cache:
    return get_qwen_input_cache[cache_key]

  inputs = tokenizer(prompt, return_tensors="pt").to(device)
  output = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample=False,
      pad_token_id=tokenizer.eos_token_id
  )
  output = output[0]
  output = output[len(inputs[0]):]
  output = tokenizer.decode(output, skip_special_tokens=True)
  get_qwen_input_cache[cache_key] = output
  return output
  

In [ ]:
# Write Your Code Here

def get_text_raw(question: str):
    messages = [
        {"role": "system", "content": "You are a careful question-answering assistant. Answer only from your internal knowledge. Keep the answer short, direct, and free of extra explanation unless the question asks for it."},
        {"role": "user", "content": question}
    ]

    prompt = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking = False
    )
    output = get_qwen_output(prompt)

    return output


sample_answer = get_text_raw("What is the capital of France?")
print(sample_answer)

if IS_TEST:
  gold_df["output_raw"] = sample_answer
else:
  gold_df["output_raw"] = gold_df["question"].apply(get_text_raw)


Paris



In [ ]:
for i in range(len(gold_df)):
  row = gold_df.iloc[i]
  print("Question:", row["question"])
  print("Answer:", row["answer"])
  print("Model:", row["output_raw"])
  print("###########")

Question: When is the last time a fumble return touchdown happened in a Super Bowl?
Answer: Super Bowl XXVIII
Model: The last time a fumble return touchdown occurred in a Super Bowl was in **Super Bowl LVIII** (played on February 11, 2024).

The Chicago Bears' return team, led by quarterback Jayden Daniels, recovered a fumble near the end of the game and returned it for
###########
Question: According to the AAA, what is the percentage of the gas stations that ran out of gasoline?
Answer: last week of February 1974,
Model: According to the American Automobile Association (AAA), approximately **10%** of gas stations ran out of gasoline.

###########
Question: What choice did French have for surrendering land?
Answer: continental North American possessions east of the Mississippi or the Caribbean islands of Guadeloupe and Martinique
Model: There is no historical record of a French choice regarding the surrender of land. France has never been a nation that surrenders territory to another 

---
## 5 RAG - ZeroShot (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, implement a <b>Zero-shot</b> RAG pipeline. For each question in <b>squad_gold_20.csv</b>,
retrieve the top 5 relevant passages from LanceDB and construct a prompt that includes the retrieved context.
The model should answer the question based only on the provided passages, without any examples or reasoning steps.
Store the predicted answers for evaluation in later sections.
You have to print the model responses.
</p>

In [ ]:
# Write Your Code Here
def get_test_rag_zshot(question: str):
  contexts = retrieve_for_gold(question)
  messages = [
      {"role": "system", "content": "You are a concise question-answering assistant. Use the provided context when it is helpful, otherwise rely on your internal knowledge. Return only the final answer."},
      {"role": "user", "content": f"Question: {question}\nContext: {contexts}\nAnswer briefly and directly."}
  ]

  prompt = qwen_tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking = False
  )
  output = get_qwen_output(prompt)
  return output

sample_answer = get_test_rag_zshot("What is the capital of France?")
print(sample_answer)

if IS_TEST:
  gold_df["output_rag_zshot"] = sample_answer
else:
  gold_df["output_rag_zshot"] = gold_df["question"].apply(get_test_rag_zshot)


Paris



In [ ]:
for i in range(len(gold_df)):
  row = gold_df.iloc[i]
  print("Question", row["question"])
  print("Answer", row["answer"])
  print("Model", row["output_rag_zshot"])
  print("###########")

Question When is the last time a fumble return touchdown happened in a Super Bowl?
Answer Super Bowl XXVIII
Model 1985

###########
Question According to the AAA, what is the percentage of the gas stations that ran out of gasoline?
Answer last week of February 1974,
Model 0%

###########
Question What choice did French have for surrendering land?
Answer continental North American possessions east of the Mississippi or the Caribbean islands of Guadeloupe and Martinique
Model France chose to cede its continental North American possessions east of the Mississippi.

###########
Question Which articles state that the member states' rights to deliver public services may not be obstructed?
Answer Articles 106 and 107
Model Article 107 of the Treaty on European Union.

###########
Question The time required to output an answer on a deterministic Turing machine is expressed as what?
Answer state transitions
Model The total number of state transitions (or steps) the machine makes before it halts

---
## 6 RAG - FewShot (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, implement a <b>Few-shot</b> RAG pipeline. For each question in <b>squad_gold_20.csv</b>,
retrieve the top 5 relevant passages from LanceDB and construct a prompt that includes the retrieved context along with <b>3 examples</b> of question-answer pairs.
The examples should guide the model on the expected format and style of the answer.
Store the predicted answers for evaluation in later sections.
You have to print the model responses.
</p>

In [ ]:
# Write Your Code Here
def get_test_rag_few_shot(user_question: str):
    contexts = retrieve_for_gold(user_question)

    example_answers = [
        "University of North Florida",
        "Andre Caldwell",
        "2p - 1"
    ]

    example_lines = []
    for i in range(3):
        ex_question, _ = dataset[i]
        example_lines.append(
            f"Example {i + 1} Question: {ex_question}\nExample {i + 1} Answer: {example_answers[i]}"
        )

    messages = [
        {"role": "system", "content": "You are a concise question-answering assistant. Use the provided context and examples to match the answer style, but return only the final answer."},
        {"role": "user", "content": f"Context: {contexts}\n\n" + "\n\n".join(example_lines) + f"\n\nQuestion: {user_question}\nAnswer briefly and directly."}
    ]

    # Apply chat template
    prompt = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking = False,
    )
    output = get_qwen_output(prompt, max_new_tokens=256)
    return output

sample_answer = get_test_rag_few_shot("What is the capital of France?")
print(sample_answer)

if IS_TEST:
  gold_df["output_rag_few_shot"] = sample_answer
else:
  gold_df["output_rag_few_shot"] = gold_df["question"].apply(get_test_rag_few_shot)


Paris



In [ ]:
for i in range(len(gold_df)):
  row = gold_df.iloc[i]
  print("Question", row["question"])
  print("Answer", row["answer"])
  print("Model", row["output_rag_few_shot"])
  print("###########")

Question When is the last time a fumble return touchdown happened in a Super Bowl?
Answer Super Bowl XXVIII
Model 1994

###########
Question According to the AAA, what is the percentage of the gas stations that ran out of gasoline?
Answer last week of February 1974,
Model 0%

###########
Question What choice did French have for surrendering land?
Answer continental North American possessions east of the Mississippi or the Caribbean islands of Guadeloupe and Martinique
Model Either its continental North American possessions east of the Mississippi or the Caribbean islands of Guadeloupe and Martinique.

###########
Question Which articles state that the member states' rights to deliver public services may not be obstructed?
Answer Articles 106 and 107
Model Articles 106 and 107

###########
Question The time required to output an answer on a deterministic Turing machine is expressed as what?
Answer state transitions
Model f(n)

###########
Question How many general questions are availabl

---
## 7 RAG - CoT (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, implement a <b>Chain-of-Thought (CoT)</b> RAG pipeline. For each question in <b>squad_gold_20.csv</b>,
retrieve the top 5 relevant passages from LanceDB and construct a prompt that guides the model to reason step by step before giving the final answer.
Store both the full reasoning output and the final extracted answer for evaluation in later sections.
You have to print the model responses.
</p>

In [ ]:
# Write Your Code Here


def get_test_rag_cot(question: str):
    contexts = retrieve_for_gold(question)

    messages = [
        {
            "role": "system",
            "content": (
                "think and find and tell steps to answer the question"
            ),
        },
        {
            "role": "user",
            "content": (
                f"Context: {contexts}\n\n"
                f"Question: {question}\n\n"
                "Provide a short answer."
            ),
        },
    ]

    # Apply chat template
    prompt = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    thinking = get_qwen_output(prompt, max_new_tokens=64)

    prompt = messages + [{"role": "assistant", "content": thinking}]
    prompt[0]['content'] = 'just answer the question, you already explained the steps.'
    prompt = qwen_tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    answer = get_qwen_output(prompt, max_new_tokens=32)
    # in order to limit thinking tokens to not swallow answer budget i manually split it
    return answer, thinking


sample_answer, sample_thinking = get_test_rag_cot("What is result of 560 / 280")
print("answer:", sample_answer)
print("think:", sample_thinking)


if IS_TEST:
  gold_df["output_rag_cot_answer"] = sample_answer
  gold_df["output_rag_cot_thinking"] = sample_thinking

else:
  gold_df["output_rag_cot"] = gold_df["question"].apply(get_test_rag_cot)
  gold_df["output_rag_cot_answer"] = gold_df["output_rag_cot"].apply(lambda x: x[0])
  gold_df["output_rag_cot_thinking"] = gold_df["output_rag_cot"].apply(lambda x: x[1])
  gold_df.drop("output_rag_cot", axis = 1, inplace = True)


answer: The result of 560 / 280 is 2.

think: There is no result to provide because the question asks for the mathematical result of "560 / 280" but the provided text contains no information about this specific calculation. The text only discusses historical events (Apollo 8, the Dutch Revolt, Martin Luther), geography (Interstate 5, San


In [ ]:
for i in range(len(gold_df)):
  row = gold_df.iloc[i]
  print("Question", row["question"])
  print("Answer", row["answer"])
  print("Model think", row["output_rag_cot_thinking"])
  print("Model answer", row["output_rag_cot_answer"])
  print("###########")

Question When is the last time a fumble return touchdown happened in a Super Bowl?
Answer Super Bowl XXVIII
Model think There is no record of a "fumble return touchdown" in a Super Bowl because the term is historically inaccurate for the sport.

Here is the breakdown of why this question cannot be answered as stated:

1.  **Terminology Error**: In American football, a "fumble return" refers to a
Model answer There is no record of a "fumble return touchdown" in a Super Bowl because the term is historically inaccurate for the sport.

In American football, a
###########
Question According to the AAA, what is the percentage of the gas stations that ran out of gasoline?
Answer last week of February 1974,
Model think There is no information in the provided text about the AAA, gas stations, or the percentage of stations that ran out of gasoline. The text only discusses Apollo 8, the Dutch Revolt, Japanese cars, and US urbanization.

Model answer The provided text does not contain the answer t

---
## 8 Evaluation with ROUGE (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, evaluate the predicted answers from all four methods (No RAG, Zero-shot, Few-shot, and CoT)
using <b>ROUGE-1</b>, <b>ROUGE-2</b>, and <b>ROUGE-L</b> metrics with the <b>rouge-score</b> library.
For each method, compute and display the <b>average score</b> across all 20 questions.
Display the final results in a comparison table to analyze the impact of each prompting strategy.
</p>

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)


def calculate_rouge_wrapper(col_name: str):
    def inner(row):
        scores = scorer.score(row["answer"], row[col_name])

        # Extract Precision, Recall, and F1 for ROUGE-1, ROUGE-2, ROUGE-L
        return pd.Series(
            {
                "rouge1_precision": scores["rouge1"].precision,
                "rouge1_recall": scores["rouge1"].recall,
                "rouge1_f1": scores["rouge1"].fmeasure,
                "rouge2_precision": scores["rouge2"].precision,
                "rouge2_recall": scores["rouge2"].recall,
                "rouge2_f1": scores["rouge2"].fmeasure,
                "rougeL_precision": scores["rougeL"].precision,
                "rougeL_recall": scores["rougeL"].recall,
                "rougeL_f1": scores["rougeL"].fmeasure,
            }
        )
    return inner

cols = (
    "output_raw",
    "output_rag_zshot",
    "output_rag_few_shot",
    "output_rag_cot_answer"
)
results_df = pd.DataFrame()
for col in cols:
  calculate_rouge = calculate_rouge_wrapper(col)

  scores_df = gold_df.apply(calculate_rouge, axis=1)
  scores_df = scores_df.mean()
  scores_df.name = col
  results_df = pd.concat([results_df, scores_df], axis=1)
results_df

,output_raw,output_rag_zshot,output_rag_few_shot,output_rag_cot_answer
rouge1_precision,0.035840,0.277605,0.410776,0.325989
rouge1_recall,0.297500,0.541667,0.550000,0.641667
rouge1_f1,0.060293,0.315243,0.437138,0.348469
rouge2_precision,0.003517,0.116653,0.335784,0.209989
rouge2_recall,0.058333,0.273333,0.400000,0.315000
rouge2_f1,0.006618,0.135053,0.351161,0.198475
rougeL_precision,0.034677,0.276135,0.409260,0.320357
rougeL_recall,0.285000,0.525000,0.533333,0.612500
rougeL_f1,0.058165,0.312540,0.434360,0.339361


---
## 8.1  Analyze ROUGE Scores (2 Points)

> **💬 Question (written answer — ~200 words):**  
> Analyze the ROUGE scores obtained from all four methods (No RAG, Zero-shot, Few-shot, and CoT).  
> - Which method achieved the highest ROUGE scores and why?  
> - How much did RAG improve the results compared to No RAG baseline?  
> - What does the difference between ROUGE-1 and ROUGE-L tell you about the quality of the generated answers?  
> - Based on your results, which prompting strategy would you recommend and why?  

- few shot had the best score. because it had better input. since our model is not that large, its knowledge is lacking. so raw had no change. and since the task is not reasoning base, cot models tend to generate long irrelevant data. between zero shot and few shot, because we provide model with examples, it knows how to answer and that increases fewshot rouge scores.
- if we consider only f1 scores, it improved scores 5 times. (comparing zshot with raw)
- rouge1 measures the overlap of 1grams but rougeL measure longest common sequence. in this question, it doesn't matter that much because the answers are a word (maybe multipart word) but not a paragraph. so the scores are near eachother.
- since fewshot has best results, it is my recommended strategy. it provides knowledge so model know the answer and then tell the model how to answer.

*Write your answer here.*

---
## 9 Evaluation with RoBERTa

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, evaluate the predicted answers from all four methods (No RAG, Zero-shot, Few-shot, and CoT)
using a <b>RoBERTa-based NLI model</b> loaded from Hugging Face (<b>FacebookAI/roberta-large-mnli</b>) to measure semantic equivalence between the predicted and gold answers.
For each predicted answer, the model classifies the relationship with the gold answer as <b>entailment</b>, <b>neutral</b>, or <b>contradiction</b>.
For each method, compute and display the <b>average percentage</b> of each label across all 20 questions.
Display the final results in a comparison table alongside the ROUGE scores to analyze the impact of each prompting strategy.
</p>

## 9.1 Load RoBERTa NLI (2 Points)

Load the RoBERTa NLI model and tokenizer from Hugging Face (`FacebookAI/roberta-large-mnli`).

In [ ]:
# Write Your Code Here

nli_model_name = "FacebookAI/roberta-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(device)



Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6746.10it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 9.2  Define NLI Scoring Function (2 Points)

Define a function that takes a list of predicted answers and gold answers, and for each pair classifies the relationship as **entailment**, **neutral**, or **contradiction**.

In [ ]:
# Write Your Code Here


def get_nli_label(premise, hypothesis):
    inputs = nli_tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = nli_model(**inputs)

    logits = outputs.logits
    predicted_class_id = logits.argmax().item()

    label = nli_model.config.id2label[predicted_class_id].upper()
    return label
row = gold_df.iloc[0]
get_nli_label(row['answer'], row['output_raw'])

'NEUTRAL'

## 9.3  Run Evaluation and Display Results (1 Points)

Run the NLI scoring function on all four methods and display the average percentage of each label in a comparison table.
Also merge the NLI results with the ROUGE scores from the previous section into a single final table.

In [41]:
# Write Your Code Here

def evaluate_method_nli(df, pred_col, gold_col):
    nli_labels = []


    for _, row in df.iterrows():
        gold = str(row[gold_col])
        pred = str(row[pred_col])

        label = get_nli_label(gold, pred)
        nli_labels.append(label)


    total = len(nli_labels)
    ent_pct = (nli_labels.count("ENTAILMENT") / total) * 100
    neu_pct = (nli_labels.count("NEUTRAL") / total) * 100
    con_pct = (nli_labels.count("CONTRADICTION") / total) * 100


    return {
        "Entailment (%)": round(ent_pct, 2),
        "Neutral (%)": round(neu_pct, 2),
        "Contradiction (%)": round(con_pct, 2),
    }


cols = {
    "output_raw",
    "output_rag_zshot",
    "output_rag_few_shot",
    "output_rag_cot_answer"
}

results = {}
for col in tqdm(cols):
    results[col] = evaluate_method_nli(gold_df, pred_col=col, gold_col = "answer")

results_df2 = pd.DataFrame(results)
results_df = pd.concat((results_df, results_df2), axis=0)

results_df

100%|██████████| 4/4 [00:04<00:00,  1.09s/it]


,output_raw,output_rag_zshot,output_rag_few_shot,output_rag_cot_answer
rouge1_precision,0.035840,0.277605,0.410776,0.325989
rouge1_recall,0.297500,0.541667,0.550000,0.641667
rouge1_f1,0.060293,0.315243,0.437138,0.348469
rouge2_precision,0.003517,0.116653,0.335784,0.209989
rouge2_recall,0.058333,0.273333,0.400000,0.315000
rouge2_f1,0.006618,0.135053,0.351161,0.198475
rougeL_precision,0.034677,0.276135,0.409260,0.320357
rougeL_recall,0.285000,0.525000,0.533333,0.612500
rougeL_f1,0.058165,0.312540,0.434360,0.339361
Entailment (%),10.000000,25.000000,50.000000,25.000000


---
## 9.4  Analyze RoBERTa NLI Results (4 Points)

> **💬 Question (written answer — ~200 words):**  
> Analyze the RoBERTa NLI results obtained from all four methods (No RAG, Zero-shot, Few-shot, and CoT).  
> - Which method achieved the highest entailment percentage and why?  
> - How does the contradiction rate change between No RAG and RAG-based methods?  
> - Comparing the NLI results with the ROUGE scores, do both metrics agree on which method performs best?  
> - What does a high neutral percentage tell you about the quality of the generated answers?  

- We can see that RAG with fewshot has the highest entailment score. it is because the model has the context and see the way it should answer. after that, we see that the rag zero shot and then raw output have the highest entailment. the reason for rag zero shot score is obvious but the reason that cot has even worse score than raw is that because the task here is not a reasoning task, cot has little advantage here. in fact, since the model has to explain itself for some memorial questions, the model generate more without answering question hence the lower score.
- we can see that contradiction almost doesn't change. so the addition of extra document don't result in model saying more correct or wrong answers, it mainly decrease the neutral percentage. however, in one of my runs where my prompting had issues, the contradiction in non rag models were higher. but it was fixable by better prompts
- except for rouge1, other metrics nearly match eachother. however at first i got results where the entailment for few shot was higher but its rouge scores (all three rouge1, rouge2 and rougeL) of zero shot was higher. it was because the zero shot nearly replicated the question/context but didn't answer the question however, fewshot answered with different words. but now with better prompting, rouge scores improved and currently it matches entailment. (except for rouge1 where cot has the highest score but lower entailment)
- when neutral percentage is high (like cot or raw) it tells us that the model didn't generate correct or wrong outputs. but the output is irrelavant. cot first neutralment was higher because i increased its max_new_tokens. but it only increased neutralment because the model didn't have neccessary knowledge to answer.


---
## 10  Fine-Tuning With LoRA



<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, you need to fine-tune the <b>Qwen/Qwen 3.5 2B</b> model using <b>LoRA (Low-Rank Adaptation)</b>.
To avoid <b>Out of Memory</b> errors, please <b>restart your runtime</b> before running this section and reinstall the required libraries from the <b>requirements.txt</b> file.
</p>

## 10.1 Load Fine-Tuning Dataset

Load the fine-tuning datasets from **finetune_train.csv** and **finetune_val.csv** and prepare them for tokenization.

In [ ]:
# Write Your Code Here
finetune_df_train = pd.read_csv("./Data/finetune_train.csv")
print(finetune_df_train.head())

finetune_df_val = pd.read_csv("./Data/finetune_val.csv")
print(finetune_df_val.head())

finetune_train_dataset = HuggingFaceDataset.from_pandas(finetune_df_train)
finetune_test_val = HuggingFaceDataset.from_pandas(finetune_df_val)

                                                text
0  Architecturally, the school has a Catholic cha...
1  As at most other universities, Notre Dame's st...
2  The university is the major seat of the Congre...
3  The College of Engineering was established in ...
4  All of Notre Dame's undergraduate students are...
                                                text
0  Super Bowl 50 was an American football game to...
1  The Panthers finished the regular season with ...
2  The Broncos took an early lead in Super Bowl 5...
3  CBS broadcast Super Bowl 50 in the U.S., and c...
4  In early 2012, NFL Commissioner Roger Goodell ...


## 10.2 Load Model and Tokenizer (2 Points)

Load the **Qwen/Qwen3.5-2B** model and its tokenizer from 🤗 Hugging Face using `AutoModelForCausalLM` and `AutoTokenizer`.

In [ ]:
# Write Your Code Here
finetune_model_name = 'qwen/qwen3.5-2b'
finetune_tokenizer = transformers.AutoTokenizer.from_pretrained(finetune_model_name)
finetune_model = transformers.AutoModelForCausalLM.from_pretrained(
    finetune_model_name,
    torch_dtype="auto",
    # device_map={"": 0}  # force one gpu (for kaggle)
).to(device)
finetune_tokenizer.pad_token = finetune_tokenizer.eos_token

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [ ]:
if FINETUNE_DONE:
  from transformers import AutoModelForCausalLM, AutoTokenizer
  from peft import PeftModel

  finetune_tokenizer = AutoTokenizer.from_pretrained("./lora_output/tokenizer")


  # Load LoRA adapter
  finetune_model = PeftModel.from_pretrained(
      finetune_model,
      "./lora_output/trainer"
  )

  finetune_model.eval()
  print("Model loaded")

Model loaded


## 10.3 Tokenize Dataset (4 Points)

Define a tokenization function using the loaded tokenizer and apply it to both the train and validation datasets.

In [ ]:
# since I used SFTTrainer and passed tokenizer to it, it is not needed

## 10.4 Fine-Tune with LoRA (8 Points)

Fine-tune the model using **LoRA** with the following configuration:
- `r=8`, `lora_alpha=16`, `lora_dropout=0.05`
- `target_modules`: `q_proj` and `v_proj`
- `num_train_epochs=1`
- `per_device_train_batch_size=2`
- `gradient_accumulation_steps=8`
- `learning_rate=2e-4`
- Enable `fp16` and `gradient_checkpointing` to avoid out of memory errors.

After training, save the fine-tuned model to **./lora_output** or GOOGLE DRIVE for use in the next sections.

In [ ]:
# Write Your Code Here
if not FINETUNE_DONE:

  finetune_model.config.use_cache = False

  peft_config = LoraConfig(
      r=8,
      lora_alpha=32,
      lora_dropout=0.05,
      target_modules=["q_proj", "v_proj"],
      task_type="CAUSAL_LM"
  )

  training_args = TrainingArguments(
      output_dir="./lora_output",
      num_train_epochs = 1,
      per_device_train_batch_size=2,
      gradient_accumulation_steps=8,
      learning_rate=2e-4,
      logging_steps=10,
      fp16=True,
      gradient_checkpointing = True,
  )

  trainer = SFTTrainer(
      model=finetune_model,
      processing_class = finetune_tokenizer,
      train_dataset=finetune_train_dataset,
      eval_dataset=finetune_test_val,
      peft_config=peft_config,
      args=training_args,
  )

  trainer.train()
  trainer.save_model("./lora_output/trainer")
  finetune_tokenizer.save_pretrained("./lora_output/tokenizer")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/4561 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4561 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4561 [00:00<?, ? examples/s]

Step,Training Loss
10,2.648213
20,2.685767
30,2.638280
40,2.580196
50,2.521692
60,2.578646
70,2.555308
80,2.495031
90,2.601590
100,2.549567


TrainOutput(global_step=286, training_loss=2.546575542930123, metrics={'train_runtime': 4002.5848, 'train_samples_per_second': 1.14, 'train_steps_per_second': 0.071, 'total_flos': 6430981819891200.0, 'train_loss': 2.546575542930123, 'entropy': 2.69426213241205, 'num_tokens': 600606.0, 'mean_token_accuracy': 0.45246326850681773, 'epoch': 1.0})

## 10.5 Generate Answers with Fine-Tuned Model (2 Points)

Using the fine-tuned model, generate answers for the 20 questions in **squad_gold_20.csv** without providing any context (No RAG).
Store the predicted answers alongside the gold answers in a CSV file.

In [ ]:
# Write Your Code Here

def get_text_finetuned(question: str):

    messages = [
        {"role": "system", "content": "You are a helpful assistant that answers questions briefly and concisely."},
        {"role": "user", "content": question}
    ]

    prompt = finetune_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    output = get_qwen_output(prompt, model = finetune_model, tokenizer=finetune_tokenizer)
    return output


sample_answer = get_text_finetuned("What is the capital of France?")
print(sample_answer)

if IS_TEST:
  gold_df["output_finetuned"] = sample_answer
else:
  gold_df["output_finetuned"] = gold_df["question"].apply(get_text_finetuned)


The capital of France is Paris.



## 10.6 Evaluate with ROUGE (2 Points)

Compute **ROUGE-1**, **ROUGE-2**, and **ROUGE-L** scores for the fine-tuned model's predictions using the **rouge-score** library.
Display the average scores across all 20 questions.

In [42]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

calculate_rouge = calculate_rouge_wrapper("output_finetuned")

scores_df = gold_df.apply(calculate_rouge, axis=1)
scores_df = scores_df.mean()
scores_df = pd.DataFrame({'finetuned': scores_df})
scores_df

,finetuned
rouge1_precision,0.045903
rouge1_recall,0.218333
rouge1_f1,0.072451
rouge2_precision,0.011565
rouge2_recall,0.065000
rouge2_f1,0.018731
rougeL_precision,0.040056
rougeL_recall,0.199583
rougeL_f1,0.063538


## 10.7 Load RoBERTa NLI Model (1 Points)

Load the **FacebookAI/roberta-large-mnli** model and its tokenizer from 🤗 Hugging Face.

In [ ]:
# Write Your Code Here
# Already done in 8.1 (reran the cell without running whole notebook)

## 10.8 Define RoBERTa NLI Scoring Function (1 Points)

Define a function that takes a list of predicted answers and gold answers, and for each pair classifies the relationship as **entailment**, **neutral**, or **contradiction**.

In [ ]:
# Write Your Code Here
# Already done (reran the cell without running whole notebook)

## 10.9 Evaluate Fine-Tuned Model with RoBERTa NLI (1 Points)

Run the NLI scoring function on the fine-tuned model's predictions and display the average percentage of each label across all 20 questions.

In [43]:
# Write Your Code Here


results ={}
results["finetuned"] = evaluate_method_nli(gold_df, pred_col="output_finetuned", gold_col = "answer")

results_df2 = pd.DataFrame(results)
results_df2 = pd.concat((scores_df, results_df2), axis=0)

results_df = pd.concat((results_df, results_df2), axis=1)
results_df

,output_raw,output_rag_zshot,output_rag_few_shot,output_rag_cot_answer,finetuned
rouge1_precision,0.035840,0.277605,0.410776,0.325989,0.045903
rouge1_recall,0.297500,0.541667,0.550000,0.641667,0.218333
rouge1_f1,0.060293,0.315243,0.437138,0.348469,0.072451
rouge2_precision,0.003517,0.116653,0.335784,0.209989,0.011565
rouge2_recall,0.058333,0.273333,0.400000,0.315000,0.065000
rouge2_f1,0.006618,0.135053,0.351161,0.198475,0.018731
rougeL_precision,0.034677,0.276135,0.409260,0.320357,0.040056
rougeL_recall,0.285000,0.525000,0.533333,0.612500,0.199583
rougeL_f1,0.058165,0.312540,0.434360,0.339361,0.063538
Entailment (%),10.000000,25.000000,50.000000,25.000000,5.000000


---
## 10.10  RAG vs Fine-Tuned Model Analysis (8 Points)

> **💬 Question (written answer — ~200 words):**  
> Compare the results of the RAG-based methods (Zero-shot, Few-shot, and CoT) with the fine-tuned model (No RAG) using both ROUGE and RoBERTa NLI scores.  
> - Which approach achieved better results overall and why?   
> - What are the limitations of fine-tuning with LoRA on a small dataset compared to RAG?  
> - In what scenarios would you prefer fine-tuning over RAG?  

- RAG achieved better score.
- On a small dataset, finetuning has little benefits. because the model either overfit or don't learn at all. so the knowledge cant be go into its parameters. also the update take larger time than using RAG (since we have to train the model.)  
- if i have a large model with a large dataset and that data is near static so it doesn't change (for at least a medium amount of time) then finetuning makes sense. because its inference cost is cheaper (when comparing with RAG) and we don't have to update it. but if data is highly dynamic or the model need to be up to date, then RAG is the more suitable approach


In [ ]:
gold_df.to_csv("./final.csv")